# 21. The collections toolbox

After this notebook you'll be able to count and group things with less code, compare and subtract counts, and use a **queue** that serves things in the order they arrived.

Nothing here is a new idea. These are better tools for jobs you already do with dictionaries and lists.

## Better kitchen tools

You can grate a carrot with a knife. It works, but a grater is faster and you're less likely to cut yourself. Python's **`collections`** is a drawer of better tools. This notebook covers three of them, and one line brings all three in:

In [ ]:
from collections import defaultdict, Counter, deque

LeetCode allows this import.

## `defaultdict`: a dictionary that fills in missing keys

### Counting, without `.get`

Here's the counting pattern you know from the dictionary notebooks:

In [ ]:
counts = {}
for ch in "banana":
    counts[ch] = counts.get(ch, 0) + 1
print(counts)

```
{'b': 1, 'a': 3, 'n': 2}
```

The `.get(ch, 0)` is there because the first time you meet a letter, it isn't a key yet. A **`defaultdict`** takes care of that for you:

In [ ]:
counts = defaultdict(int)
for ch in "banana":
    counts[ch] += 1
print(counts)
print(counts["a"])
print(dict(counts))

```
defaultdict(<class 'int'>, {'b': 1, 'a': 3, 'n': 2})
3
{'b': 1, 'a': 3, 'n': 2}
```

When you ask a `defaultdict` for a key that isn't there, it doesn't stop with `KeyError`. It **adds** the key with a starting value, then carries on. The thing in the brackets says what the starting value is:

| You write | Starting value for a new key |
| --- | --- |
| `defaultdict(int)` | `0` |
| `defaultdict(list)` | `[]`, an empty list |
| `defaultdict(set)` | `set()`, an empty set |

So `counts[ch] += 1` works even for a brand-new letter: it starts at 0, then adds 1.

It prints with `defaultdict(<class 'int'>, ...)` in front. Everything else works like a normal dictionary: `in`, `.items()`, `len`, looping. Wrap it in `dict(...)` to get a normal dictionary back.

### Grouping with `defaultdict(list)`

Put words into groups by their first letter. With a plain dictionary you must make each empty list yourself before you can `append`. With `defaultdict(list)`, a new key starts as an empty list, so you can `append` straight away.

The `print` inside the loop shows the groups growing.

In [ ]:
words = ["apple", "banana", "avocado", "cherry", "blueberry"]

groups = defaultdict(list)
for w in words:
    groups[w[0]].append(w)
    print(w, "->", dict(groups))

```
apple -> {'a': ['apple']}
banana -> {'a': ['apple'], 'b': ['banana']}
avocado -> {'a': ['apple', 'avocado'], 'b': ['banana']}
cherry -> {'a': ['apple', 'avocado'], 'b': ['banana'], 'c': ['cherry']}
blueberry -> {'a': ['apple', 'avocado'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}
```

When `"apple"` arrived, there was no `'a'` key, so `groups['a']` made one as `[]`, and `append` put `"apple"` in. When `"avocado"` arrived, `'a'` was already there, so it joined the same list.

The **key** is what you group **by** (the first letter). The **value** is the list of everything in that group.

### Grouping pairs: students by section

Often the data comes as pairs. Here each pair is a student and their section:

In [ ]:
students = [("Tanvi", "10A"), ("Riya", "10B"), ("Aman", "10A"), ("Kabir", "10C"), ("Zoya", "10B")]

by_section = defaultdict(list)
for name, section in students:
    by_section[section].append(name)

for section in sorted(by_section):
    print(section, "->", by_section[section])

```
10A -> ['Tanvi', 'Aman']
10B -> ['Riya', 'Zoya']
10C -> ['Kabir']
```

`for name, section in students:` unpacks each pair, like tuples in notebook 12. `sorted(by_section)` gives the keys in order.

`defaultdict(set)` works the same way, when you don't want repeats in a group. Use `.add` instead of `.append`.

**Try it:** group the numbers `[1, 2, 3, 4, 5, 6]` into `"even"` and `"odd"` with `defaultdict(list)`.

## `Counter`: counting in one line

`Counter` counts everything in a string or a list, in one go, and adds a few extra tools.

In [ ]:
c = Counter("mississippi")
print(c)
print(c["s"])
print(c["z"])
print(c.most_common(2))
print(Counter([3, 1, 3, 3, 2]))

```
Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})
4
0
[('i', 4), ('s', 4)]
Counter({3: 3, 1: 1, 2: 1})
```

- `c["s"]` reads a count, like a dictionary.
- `c["z"]` is `0`, not an error. A letter that never appeared has count 0. (And unlike `defaultdict`, reading a missing key does **not** add it.)
- **`most_common(k)`** gives the `k` most common items, biggest first, as a list of `(item, count)` tuples. So `c.most_common(1)[0][0]` is the most common item itself: the first tuple, then its first part.

### Same letters? Compare two Counters

Two words are **anagrams** when they use exactly the same letters, the same number of times ("listen" and "silent"). Two Counters are equal (`==`) when every count matches. **Predict first**, then run:

In [ ]:
print(Counter("listen") == Counter("silent"))
print(Counter("apple") == Counter("plea"))

```
True
False
```

"apple" has two `p`s and "plea" has one, so they're not equal.

### What's missing? Subtract two Counters

You want to make the word `"banana"`, and you have the letter tiles `"bnaxa"`. Which tiles are you short of?

In [ ]:
need = Counter("banana")
have = Counter("bnaxa")
print(need - have)
print(have - need)
print(Counter("ab") - Counter("abc"))

```
Counter({'a': 1, 'n': 1})
Counter({'x': 1})
Counter()
```

| Letter | need | have | need − have |
| --- | --- | --- | --- |
| `b` | 1 | 1 | 0, left out |
| `a` | 3 | 2 | **1** |
| `n` | 2 | 1 | **1** |
| `x` | 0 | 1 | below 0, left out |

Anything that drops to 0 or below is left out of the answer. So `need - have` shows only what you're short of. If you're short of nothing, you get an empty `Counter()`, which counts as `False` in an `if`. That's how you check "can I make this word?": `if not (need - have):`.

## `deque`: a queue with two open ends

### A queue at a ticket counter

At a ticket counter, new people join at the **back**, and the person at the **front** is served first. First in, first out. That's a **queue**: the opposite of the pile of plates (the stack) in notebook 14, where the last one in was the first one out.

A **`deque`** (say "deck"; short for "double-ended queue") is a list-like row where you can add and take away at **both** ends quickly.

In [ ]:
q = deque([1, 2, 3])
q.append(4)
q.appendleft(0)
print(q)
print(q.popleft())
print(q.pop())
print(q)
print(q[0], len(q))

```
deque([0, 1, 2, 3, 4])
0
4
deque([1, 2, 3])
1 3
```

```
            appendleft(x)                 append(x)
                 │                            │
                 ▼                            ▼
   front ──►  [ 0,   1,   2,   3,   4 ]  ◄── back
                 │                            │
                 ▼                            ▼
             popleft()                      pop()
```

For a queue you use **`append`** (join at the back) and **`popleft`** (serve the front). `q[0]`, `len(q)` and `while q:` all work like a list.

### Why not a list?

A list can do `pop(0)` and give the right answer. But a list keeps its items in numbered boxes. Take the item out of box 0, and every other item has to shift one box to the left. With 100,000 items, that's 99,999 moves for every single `pop(0)`. A deque doesn't renumber anything.

You can see the difference. This cell empties a list of 100,000 numbers with `pop(0)`, then a deque with `popleft()`, and times both. (`time.time()` reads the clock, in seconds. You don't need it for anything else.)

In [ ]:
import time

nums = list(range(100_000))
start = time.time()
while nums:
    nums.pop(0)
print("list with pop(0):   ", round(time.time() - start, 3), "seconds")

q = deque(range(100_000))
start = time.time()
while q:
    q.popleft()
print("deque with popleft():", round(time.time() - start, 3), "seconds")

Your numbers will be a bit different, but something like `0.5` seconds for the list and `0.002` for the deque: hundreds of times faster. On LeetCode, a slow answer fails with **Time Limit Exceeded**, even when it's right. So: for a stack, a list is fine. For a queue, use a deque.

### Putting it together: a ticket counter

Three people are waiting. Each person buys **one** ticket per turn. If they need more, they go to the back of the queue and wait again. Each item in the queue is a tuple: `(name, tickets still needed)`.

**Predict first:** who finishes last?

In [ ]:
queue = deque([("Tanvi", 2), ("Riya", 1), ("Aman", 3)])
turn = 0

while queue:
    name, needed = queue.popleft()
    turn += 1
    needed -= 1
    print("turn", turn, ":", name, "buys a ticket, needs", needed, "more")
    if needed > 0:
        queue.append((name, needed))
    print("        waiting now:", list(queue))

```
turn 1 : Tanvi buys a ticket, needs 1 more
        waiting now: [('Riya', 1), ('Aman', 3), ('Tanvi', 1)]
turn 2 : Riya buys a ticket, needs 0 more
        waiting now: [('Aman', 3), ('Tanvi', 1)]
turn 3 : Aman buys a ticket, needs 2 more
        waiting now: [('Tanvi', 1), ('Aman', 2)]
turn 4 : Tanvi buys a ticket, needs 0 more
        waiting now: [('Aman', 2)]
turn 5 : Aman buys a ticket, needs 1 more
        waiting now: [('Aman', 1)]
turn 6 : Aman buys a ticket, needs 0 more
        waiting now: []
```

1. `while queue:` keeps going until nobody is left. An empty deque counts as `False`.
2. `queue.popleft()` takes the front tuple, and `name, needed = ...` unpacks it.
3. If they still need tickets, `queue.append((name, needed))` puts them behind everyone already waiting. The outer brackets are the call to `append`; the inner ones make the tuple.

Everyone gets a turn before anyone gets a second turn. Aman needs three, so he finishes last, on turn 6.

**Try it:** give Riya 4 tickets and predict the order before you run it.

## Common mistakes

**Reading a missing key in a `defaultdict` adds it.**

In [ ]:
counts = defaultdict(int)
counts["a"] += 1
if counts["z"] == 0:
    print("no z")
print(dict(counts))

```
no z
{'a': 1, 'z': 0}
```

Only looking at `counts["z"]` created the key `"z"`. To check without adding, use `in`: `"z" in counts`.

**Taking from an empty queue.**

In [ ]:
q = deque()
q.popleft()

```
IndexError: pop from an empty deque
```

Use `while queue:` so you only take from the queue while there's something in it.

**Using `list.pop(0)` for a queue.** It gives the right answer, so nothing warns you, but on a big input LeetCode says **Time Limit Exceeded**. Use a deque and `popleft()`.

---

# ✏️ Your turn

Write your code in the empty cells, then run the check cell under each one.

### Task 1: count the letters

Write `count_letters(word)` that returns a normal dictionary of how many times each letter appears. Use `defaultdict(int)`, and turn it into a normal dictionary with `dict(...)` before you return it.

In [ ]:
def count_letters(word):
    # your code here
    pass

In [ ]:
from checks import check
check(count_letters("banana"), {"b": 1, "a": 3, "n": 2})
check(count_letters("tea"), {"t": 1, "e": 1, "a": 1})
check(count_letters(""), {})

### Task 2: group words by length

Write `group_by_length(words)` that returns a dictionary: each key is a length, and its value is the list of words with that length, in the order they came.

`["cat", "dog", "tiger", "ox"]` → `{3: ["cat", "dog"], 5: ["tiger"], 2: ["ox"]}`

In [ ]:
def group_by_length(words):
    # your code here
    pass

In [ ]:
check(group_by_length(["cat", "dog", "tiger", "ox"]), {3: ["cat", "dog"], 5: ["tiger"], 2: ["ox"]})
check(group_by_length(["hi"]), {2: ["hi"]})
check(group_by_length([]), {})

### Task 3: the most common letter

Write `most_common_letter(word)` that returns the letter that appears most often. Use `Counter` and `most_common`.

In [ ]:
def most_common_letter(word):
    # your code here
    pass

In [ ]:
check(most_common_letter("banana"), "a")
check(most_common_letter("hello"), "l")
check(most_common_letter("zzz"), "z")

### Task 4: LeetCode 242, Valid Anagram

Return `True` if `t` uses exactly the same letters as `s`, the same number of times. Otherwise `False`.

In [ ]:
class Solution:
    def isAnagram(self, s: str, t: str) -> bool:
        # your code here
        pass

In [ ]:
check(Solution().isAnagram("anagram", "nagaram"), True)
check(Solution().isAnagram("rat", "car"), False)
check(Solution().isAnagram("a", "ab"), False)

### Task 5: LeetCode 383, Ransom Note

You want to write `ransomNote` by cutting letters out of `magazine`. Each letter in the magazine can be used only once. Return `True` if you can make the note, `False` if not.

- `"aa"` from `"aab"` → `True`
- `"aa"` from `"ab"` → `False` (only one `a`)

Hint: "what am I short of?"

In [ ]:
class Solution:
    def canConstruct(self, ransomNote: str, magazine: str) -> bool:
        # your code here
        pass

In [ ]:
check(Solution().canConstruct("a", "b"), False)
check(Solution().canConstruct("aa", "ab"), False)
check(Solution().canConstruct("aa", "aab"), True)
check(Solution().canConstruct("", "abc"), True)

### Task 6: LeetCode 2073, Time Needed to Buy Tickets

People stand in a queue. `tickets[i]` is how many tickets person `i` wants. Each person takes **1 second** to buy **one** ticket, then goes to the back of the queue if they need more (or leaves if they're done). This is the ticket counter from above.

Return how many seconds pass until person `k` (counting from 0) has bought all their tickets.

Example: `tickets = [2, 3, 2]`, `k = 2` → `6`. The turns go: person 0, 1, 2, 0, 1, 2, and person 2 finishes on the 6th second.

Hint: put `(person, tickets needed)` tuples in a deque. Count the seconds. Stop when person `k` finishes.

In [ ]:
class Solution:
    def timeRequiredToBuy(self, tickets: list[int], k: int) -> int:
        # your code here
        pass

In [ ]:
check(Solution().timeRequiredToBuy([2, 3, 2], 2), 6)
check(Solution().timeRequiredToBuy([5, 1, 1, 1], 0), 8)
check(Solution().timeRequiredToBuy([1], 0), 1)

### Task 7 (a stretch): LeetCode 933, Number of Recent Calls, the fast way

In notebook 19 you kept every ping in a list and counted the new enough ones each time. Do it again with a deque: on each `ping(t)`, add `t` at the back, then throw away times from the **front** while they're older than `t - 3000`. What's left in the deque is the answer: its length.

Example: `ping(1)` → 1, `ping(100)` → 2, `ping(3001)` → 3, `ping(3002)` → 3.

In [ ]:
class RecentCounter:
    def __init__(self):
        # your code here
        pass

    def ping(self, t: int) -> int:
        # your code here
        pass

In [ ]:
rc = RecentCounter()
check(rc.ping(1), 1)
check(rc.ping(100), 2)
check(rc.ping(3001), 3)
check(rc.ping(3002), 3)
check(rc.ping(7000), 1)

---

# ✅ Quiz: check yourself

Put your letter between the quotes, then run the check cell under it.

**Question 1.** What does `Counter("hello")["x"]` give?

- A) `0`
- B) An error
- C) `None`

In [ ]:
answer_1 = "?"

In [ ]:
from checks import check_answer
check_answer("21-q1", answer_1)

**Question 2.** A deque holds `[5, 6, 7]`. You run `append(8)` and then `popleft()`. What comes out?

- A) `8`
- B) `5`
- C) `7`

In [ ]:
answer_2 = "?"

In [ ]:
check_answer("21-q2", answer_2)

**Question 3.** Why use a deque for a queue instead of a list with `pop(0)`?

- A) `pop(0)` is slow on a big list
- B) `pop(0)` gives the wrong answer

In [ ]:
answer_3 = "?"

In [ ]:
check_answer("21-q3", answer_3)

**Question 4.** `counts = defaultdict(int)` has no key `"z"`. What does `print(counts["z"])` do?

- A) `KeyError`
- B) Prints `0`, and `counts` stays the same
- C) Prints `0`, and adds `"z"` to `counts`

In [ ]:
answer_4 = "?"

In [ ]:
check_answer("21-q4", answer_4)

---

## Done

You can now count with `defaultdict(int)` or `Counter`, group with `defaultdict(list)`, compare and subtract counts, and run a first-in, first-out queue with a deque.

Stuck on a task? The answers are in `solutions/21-collections.ipynb`. Try for real first.

**Next:** a pile that always gives you the smallest item, and a fast way to find a place in a sorted list (`22-heapq-and-bisect`).